## Look at data


In [ ]:
from importlib import reload
from pathlib import Path

import polars as pl
from polars import selectors as cs

from ctm_ai_eval.utils import paths, plots

plots.set_template()
reload(paths)

print(paths.ExperimentPaths("?", root=Path("../tmp")).available_setups())
PATHS = paths.ExperimentPaths("query_efdbf37d", root=Path("../tmp/"))

target_names = PATHS.available_targets()
IGNORE_TARGETS = {"exact00", "exact03", "dummy"}

target_dfs = {
    k: pl.read_parquet(PATHS.haystack_target_res(k))
    for k in target_names
    if k not in IGNORE_TARGETS
}
for k, v in target_dfs.items():
    print(f"{k}: {v.shape}")

In [ ]:
df_all = pl.concat([v.with_columns(target=pl.lit(k)) for k, v in target_dfs.items()])
df_avg = (
    df_all.group_by("target")
    .agg(cs.float().mean())
    .sort("target")
    .with_columns(pl.col("target").str.replace(r"^faiss_", ""))
)
display(df_avg)

In [ ]:
reload(plots)
plots.scatter_recall_rr(df_avg)

In [ ]:
reload(plots)
plots.rag_target_recall_avg(df_avg).show()
plots.reciprocal_rank_bar(df_avg.sort("rr_chunk"), keys=("chunk",)).show()
